# Chapter 3 -- Tools as Interfaces (Practice)

Work through this notebook **after reading** `notes/ch03-tools-as-interfaces.md`. This chapter's central claim: a tool schema is a user interface whose user is a model, and most "the agent is dumb" bugs are tool-design bugs, not model-capability bugs.

You will see a deliberately **bad** tool set and a deliberately **good** tool set built against the exact same underlying task data, then fix three more broken tool designs yourself, implement a truncate-with-cursor return value (notes Section 4), and implement a validation-repair loop (notes Section 5). A final section measures the real token cost of bad vs. good return values, and an optional section runs both tool sets against a real Claude model.

Every exercise below is verified **offline** -- no API key needed to complete any of them.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import anthropic


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable the real-model sections.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_NAME = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-sonnet-5")


def test_connection(client, model_name):
    """Send a trivial ping to confirm the Bedrock connection actually works."""
    print(f"Testing connection to Bedrock (model={model_name})...")
    try:
        response = client.messages.create(
            model=model_name, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = next((b.text for b in response.content if b.type == "text"), "")
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The rest of this notebook still works fully offline -- this cell")
        print("only matters for the optional real-model section at the end.")


if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env -- skipping connection test.")
    print("The rest of this notebook still works fully offline.")
else:
    client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    test_connection(client, MODEL_NAME)


## The Shared Task and Data

Every tool below -- bad and good alike -- reads from the exact same in-memory task tracker. Holding the underlying data fixed is what makes the bad-vs-good comparison fair: nothing about *what's possible* changes, only how the interface exposes it. `REFERENCE_DATE` is a fixed "as of" date so "overdue" has a stable, reproducible meaning.

In [ ]:
REFERENCE_DATE = "2026-08-01"  # fixed "as of" date, so "overdue" is reproducible

TASK_STORE = [
    {"id": "task_1",  "title": "Fix login bug",           "assignee": "alex",  "status": "open",   "priority": "high",     "due_date": "2026-07-15", "project": "auth"},
    {"id": "task_2",  "title": "Update docs",              "assignee": "priya", "status": "closed", "priority": "low",      "due_date": "2026-06-01", "project": "docs"},
    {"id": "task_3",  "title": "Migrate database",         "assignee": "sam",   "status": "open",   "priority": "critical", "due_date": "2026-08-05", "project": "infra"},
    {"id": "task_4",  "title": "Design new logo",          "assignee": "priya", "status": "closed", "priority": "medium",   "due_date": "2026-05-20", "project": "marketing"},
    {"id": "task_5",  "title": "Refactor auth module",     "assignee": "alex",  "status": "open",   "priority": "high",     "due_date": "2026-07-20", "project": "auth"},
    {"id": "task_6",  "title": "Write onboarding guide",   "assignee": "sam",   "status": "closed", "priority": "low",      "due_date": "2026-06-15", "project": "docs"},
    {"id": "task_7",  "title": "Fix payment bug",          "assignee": "alex",  "status": "open",   "priority": "critical", "due_date": "2026-07-01", "project": "billing"},
    {"id": "task_8",  "title": "Plan Q3 roadmap",          "assignee": "priya", "status": "open",   "priority": "medium",   "due_date": "2026-08-10", "project": "planning"},
    {"id": "task_9",  "title": "Security audit",           "assignee": "sam",   "status": "open",   "priority": "critical", "due_date": "2026-07-25", "project": "infra"},
    {"id": "task_10", "title": "Update dependencies",      "assignee": "alex",  "status": "closed", "priority": "low",      "due_date": "2026-06-10", "project": "infra"},
    {"id": "task_11", "title": "Customer interview notes", "assignee": "priya", "status": "closed", "priority": "medium",   "due_date": "2026-05-30", "project": "research"},
    {"id": "task_12", "title": "Fix flaky test suite",     "assignee": "sam",   "status": "open",   "priority": "high",     "due_date": "2026-07-18", "project": "infra"},
    {"id": "task_13", "title": "Redesign settings page",   "assignee": "priya", "status": "open",   "priority": "medium",   "due_date": "2026-08-15", "project": "ui"},
    {"id": "task_14", "title": "Rotate API keys",          "assignee": "alex",  "status": "open",   "priority": "high",     "due_date": "2026-07-10", "project": "security"},
    {"id": "task_15", "title": "Archive old logs",         "assignee": "sam",   "status": "closed", "priority": "low",      "due_date": "2026-06-20", "project": "infra"},
    {"id": "task_16", "title": "Draft press release",      "assignee": "priya", "status": "open",   "priority": "low",      "due_date": "2026-08-20", "project": "marketing"},
]

print(f"TASK_STORE loaded: {len(TASK_STORE)} tasks, as of {REFERENCE_DATE}")
open_count = sum(1 for t in TASK_STORE if t["status"] == "open")
overdue_count = sum(1 for t in TASK_STORE if t["status"] == "open" and t["due_date"] < REFERENCE_DATE)
print(f"  {open_count} open, {overdue_count} of those overdue")


## The Bad Tool Set

Three violations, one per tool, each mapped to a notes section:

1. `do_thing` -- an ambiguous name and a description that says nothing (Section 2), plus a needlessly nested `opts.filters.meta.who` argument shape (Section 2's "argument shape" guidance) instead of flat parameters.
2. `dump_tasks` -- no filter parameters at all; it always returns every field of every task as raw JSON, with no truncation (Section 4 and Section 8's token-bill argument).
3. `get_task_status` -- on a bad ID, returns `"Error: 400"`, which tells the model nothing about what a valid ID looks like or what to do next (Section 5).

These are given -- nothing to implement here, just read them and notice the violations before seeing the fix.

In [ ]:
import json


def _bad_do_thing(x=None, opts=None):
    """Ambiguous name, ambiguous arg 'x', and a needlessly nested opts shape."""
    opts = opts or {}
    meta = opts.get("filters", {}).get("meta", {})
    who = meta.get("who")
    state = meta.get("state")
    matches = [
        t for t in TASK_STORE
        if (who is None or t["assignee"] == who) and (state is None or t["status"] == state)
    ]
    return json.dumps(matches)  # full records, unfiltered fields, no truncation


def _bad_dump_tasks():
    """No parameters at all -- always returns the entire store."""
    return json.dumps(TASK_STORE)


def _bad_get_task_status(id=None):
    """Unhelpful error on a bad ID -- tells the model nothing to act on."""
    task = next((t for t in TASK_STORE if t["id"] == id), None)
    if task is None:
        return "Error: 400"
    return task["status"]


BAD_DISPATCH = {
    "do_thing": _bad_do_thing,
    "dump_tasks": _bad_dump_tasks,
    "get_task_status": _bad_get_task_status,
}

BAD_TOOL_SCHEMAS = [
    {
        "name": "do_thing",
        "description": "Does the thing.",
        "input_schema": {
            "type": "object",
            "properties": {
                "x": {"type": "string"},
                "opts": {
                    "type": "object",
                    "properties": {
                        "filters": {
                            "type": "object",
                            "properties": {
                                "meta": {
                                    "type": "object",
                                    "properties": {
                                        "who": {"type": "string"},
                                        "state": {"type": "string"},
                                    },
                                }
                            },
                        }
                    },
                },
            },
            "required": [],
        },
    },
    {
        "name": "dump_tasks",
        "description": "Returns tasks.",
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
    {
        "name": "get_task_status",
        "description": "Gets status.",
        "input_schema": {
            "type": "object",
            "properties": {"id": {"type": "string"}},
            "required": ["id"],
        },
    },
]

print("Bad tool set defined:", [t["name"] for t in BAD_TOOL_SCHEMAS])


## The Good Tool Set

The same capability, redesigned:

* `search_tasks` -- one consolidated tool (Section 3) replacing `do_thing` and `dump_tasks`. Flat parameters (`assignee`, `status`, `priority`, `cursor`), a description that states the pagination contract up front, and a `response_format` switch between a lean `"concise"` string per task and a full `"detailed"` record (Section 4's verbosity-as-a-parameter). Pages at 5 results and returns an explicit `cursor` plus a message when more remain -- never a silent truncation.
* `get_task` -- replaces `get_task_status`'s bad-ID case with an actionable error: it names the bad ID, shows what a valid one looks like, and tells the model which tool to call to find the right one (Section 5).

In [ ]:
PAGE_SIZE = 5


def _good_search_tasks(assignee=None, status=None, priority=None, cursor=0, response_format="concise"):
    """Consolidated, flat, paginated, verbosity-aware search over TASK_STORE."""
    def status_matches(t):
        if status is None:
            return True
        if status == "overdue":
            return t["status"] == "open" and t["due_date"] < REFERENCE_DATE
        return t["status"] == status

    matches = [
        t for t in TASK_STORE
        if (assignee is None or t["assignee"] == assignee)
        and status_matches(t)
        and (priority is None or t["priority"] == priority)
    ]
    page = matches[cursor:cursor + PAGE_SIZE]

    if response_format == "detailed":
        page_repr = [dict(t) for t in page]
    else:
        page_repr = [f"{t['id']}: {t['title']} (due {t['due_date']})" for t in page]

    result = {"results": page_repr}
    remaining = len(matches) - (cursor + len(page))
    if remaining > 0:
        next_cursor = cursor + len(page)
        result["cursor"] = next_cursor
        result["message"] = f"{remaining} more match(es). Call again with cursor={next_cursor} to continue."
    else:
        result["message"] = f"{len(matches)} total match(es) -- all returned."
    return json.dumps(result)


def _good_get_task(task_id=None):
    """Actionable error on a bad ID instead of an opaque code."""
    task = next((t for t in TASK_STORE if t["id"] == task_id), None)
    if task is None:
        sample_ids = ", ".join(t["id"] for t in TASK_STORE[:3])
        return json.dumps({
            "error": (
                f"No task with id '{task_id}'. Valid IDs look like "
                f"{sample_ids}, .... Use search_tasks() to find the right ID."
            )
        })
    return json.dumps(task)


GOOD_DISPATCH = {
    "search_tasks": _good_search_tasks,
    "get_task": _good_get_task,
}

GOOD_TOOL_SCHEMAS = [
    {
        "name": "search_tasks",
        "description": (
            "Search tasks by assignee, status ('open', 'closed', or 'overdue'), "
            "and/or priority. Returns up to 5 concise matches per call; if more "
            "exist, the response includes a 'cursor' -- pass it back in a "
            "follow-up call to see the rest. Only set response_format='detailed' "
            "if you need a task's full record (e.g. before updating it) -- "
            "'concise' (the default) is enough for reporting or summarizing."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "assignee": {"type": "string", "description": "Exact assignee name, e.g. 'alex'"},
                "status": {"type": "string", "enum": ["open", "closed", "overdue"]},
                "priority": {"type": "string", "enum": ["low", "medium", "high", "critical"]},
                "cursor": {"type": "integer", "description": "Offset from a previous response; omit on the first call"},
                "response_format": {"type": "string", "enum": ["concise", "detailed"]},
            },
            "required": [],
        },
    },
    {
        "name": "get_task",
        "description": (
            "Get the full record for exactly one task by its ID (e.g. 'task_7'). "
            "If you don't already know the ID, call search_tasks first."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"task_id": {"type": "string", "description": "e.g. 'task_7'"}},
            "required": ["task_id"],
        },
    },
]

print("Good tool set defined:", [t["name"] for t in GOOD_TOOL_SCHEMAS])


## Exercise 1 -- Rewrite Three Bad Tool Schemas

`BAD_SCHEMAS_TO_FIX` below has three more schemas, each with a different Section-2 violation: a name+description that say nothing, an unnecessarily nested argument, and a cryptic single-letter parameter with no pagination hint. Write a corrected version of each as `GOOD_SCHEMA_1`, `GOOD_SCHEMA_2`, `GOOD_SCHEMA_3`.

Your rewrites don't need to be implemented -- an implementation already exists in `GOOD_DISPATCH` conceptually via `search_tasks`/`get_task` above. This exercise is purely about the *interface*: descriptive names, flat arguments, and descriptions long enough to actually tell the model something it couldn't infer from the name alone.

In [ ]:
BAD_SCHEMAS_TO_FIX = [
    {
        "name": "get",
        "description": "Gets stuff",
        "input_schema": {
            "type": "object",
            "properties": {"id": {"type": "string"}},
            "required": ["id"],
        },
    },
    {
        "name": "upd",
        "description": "Updates.",
        "input_schema": {
            "type": "object",
            "properties": {
                "data": {
                    "type": "object",
                    "properties": {"payload": {"type": "object"}},
                }
            },
            "required": ["data"],
        },
    },
    {
        "name": "srch",
        "description": "search",
        "input_schema": {
            "type": "object",
            "properties": {"q": {"type": "string"}},
            "required": ["q"],
        },
    },
]

# TODO 1: rewrite BAD_SCHEMAS_TO_FIX[0] ("get") -- give it a descriptive name
# and a description of at least 40 characters explaining what it returns.
GOOD_SCHEMA_1 = None

# TODO 2: rewrite BAD_SCHEMAS_TO_FIX[1] ("upd") -- flatten the nested
# data.payload argument into top-level properties, and write a real
# description (40+ characters).
GOOD_SCHEMA_2 = None

# TODO 3: rewrite BAD_SCHEMAS_TO_FIX[2] ("srch") -- rename the cryptic 'q'
# parameter to something descriptive (not 'q'), and write a description
# (40+ characters) that mentions what a caller gets back.
GOOD_SCHEMA_3 = None


In [ ]:
def _is_flat(schema):
    """True if no property in input_schema.properties is itself type 'object'."""
    props = schema["input_schema"].get("properties", {})
    return all(p.get("type") != "object" for p in props.values())


assert GOOD_SCHEMA_1 is not None, "GOOD_SCHEMA_1 is still None -- fill in TODO 1"
assert GOOD_SCHEMA_2 is not None, "GOOD_SCHEMA_2 is still None -- fill in TODO 2"
assert GOOD_SCHEMA_3 is not None, "GOOD_SCHEMA_3 is still None -- fill in TODO 3"

assert GOOD_SCHEMA_1["name"] != "get", "GOOD_SCHEMA_1 still has the original ambiguous name 'get'"
assert len(GOOD_SCHEMA_1["description"]) >= 40, "GOOD_SCHEMA_1's description is too short to say anything useful"

assert GOOD_SCHEMA_2["name"] != "upd", "GOOD_SCHEMA_2 still has the original ambiguous name 'upd'"
assert _is_flat(GOOD_SCHEMA_2), "GOOD_SCHEMA_2 still has a nested object property -- flatten it"
assert len(GOOD_SCHEMA_2["description"]) >= 40, "GOOD_SCHEMA_2's description is too short"

assert GOOD_SCHEMA_3["name"] != "srch", "GOOD_SCHEMA_3 still has the original ambiguous name 'srch'"
assert "q" not in GOOD_SCHEMA_3["input_schema"]["properties"], "GOOD_SCHEMA_3 still uses the cryptic parameter name 'q'"
assert len(GOOD_SCHEMA_3["description"]) >= 40, "GOOD_SCHEMA_3's description is too short"

print("Exercise 1 PASSED -- all three schemas rewritten with descriptive names,")
print("flat arguments, and real descriptions.")


## Exercise 2 -- Truncate-With-Cursor Return Value

`LOG_LINES` below is a small log with 40 entries, 25 of which contain `"ERROR"`. Implement `search_logs_paginated(query, cursor=0, page_size=10)` following notes Section 4: never silently truncate. Return a dict with:

* `"results"`: up to `page_size` matching lines starting at `cursor`
* `"next_cursor"`: the offset to continue from, or `None` if nothing remains
* `"message"`: a string that says how many more matches remain and how to continue when `next_cursor` is not `None`, or a completion message when it is `None`

In [ ]:
LOG_LINES = [f"{i:03d} INFO  service started ok" for i in range(15)]
LOG_LINES += [f"{i:03d} ERROR connection timeout on attempt {i}" for i in range(15, 40)]
LOG_LINES.sort()  # interleave INFO/ERROR lines like a real log would be


def search_logs_paginated(query, cursor=0, page_size=10):
    """
    Search LOG_LINES for lines containing `query`, paginated.

    Returns: {"results": [...], "next_cursor": int|None, "message": str}
    Math note: with M total matches and page_size P, there are ceil(M/P)
    pages; this call returns page starting at `cursor` and reports how
    many of the M matches remain after it.
    """
    # TODO: find every line in LOG_LINES containing `query` (case-sensitive
    # substring match is fine), slice out the page starting at `cursor` of
    # length `page_size`, and build the result dict described above.
    raise NotImplementedError("TODO: implement search_logs_paginated")


In [ ]:
page_1 = search_logs_paginated("ERROR", cursor=0, page_size=10)
assert len(page_1["results"]) == 10, f"expected 10 results on page 1, got {len(page_1['results'])}"
assert page_1["next_cursor"] == 10, f"expected next_cursor=10, got {page_1['next_cursor']}"
assert "15" in page_1["message"], "message should mention the 15 remaining matches"

page_2 = search_logs_paginated("ERROR", cursor=page_1["next_cursor"], page_size=10)
assert len(page_2["results"]) == 10, f"expected 10 results on page 2, got {len(page_2['results'])}"
assert page_2["next_cursor"] == 20

page_3 = search_logs_paginated("ERROR", cursor=page_2["next_cursor"], page_size=10)
assert len(page_3["results"]) == 5, f"expected 5 results on the final page, got {len(page_3['results'])}"
assert page_3["next_cursor"] is None, "next_cursor should be None once nothing remains"
assert "25" in page_3["message"], "final message should mention the total of 25 matches"

all_results = page_1["results"] + page_2["results"] + page_3["results"]
assert len(all_results) == 25, f"expected all 25 ERROR lines collected across pages, got {len(all_results)}"
assert len(set(all_results)) == 25, "pages overlapped or duplicated a result"

print("Exercise 2 PASSED -- 25 ERROR lines correctly paginated across 3 calls,")
print("with an explicit cursor and remaining-count message at every step.")


## Exercise 3 -- A Validation-Repair Loop

`CREATE_TASK_SCHEMA` below describes valid input for a hypothetical `create_task` tool: `priority` must be one of a fixed enum, and `due_date` must match `YYYY-MM-DD`. Implement `validate_tool_input(input_dict, schema)` so that it returns `(True, None)` for valid input, or `(False, error_message)` for invalid input -- and per notes Section 5, `error_message` must state *what* was wrong and *what a correct value looks like*, not just that something failed.

The cell after your implementation scripts a 2-attempt repair loop: attempt 1 is deliberately invalid, attempt 2 is a corrected version a model could plausibly produce *after reading your error message* -- standing in for the real self-correction Chapter 2's loop relies on.

In [ ]:
import re

CREATE_TASK_SCHEMA = {
    "required": ["title", "priority", "due_date"],
    "priority_enum": ["low", "medium", "high", "critical"],
    "due_date_pattern": r"^\d{4}-\d{2}-\d{2}$",
}


def validate_tool_input(input_dict, schema):
    """
    Validate `input_dict` against `schema`.

    Returns (True, None) if valid.
    Returns (False, error_message) if invalid, where error_message names
    the offending field, states why it's wrong, and states what a valid
    value looks like (notes Section 5 -- an error is an instruction for
    the next retry, not a diagnostic for a human).
    """
    # TODO: check every field in schema["required"] is present in
    # input_dict; if "priority" is present, check it's one of
    # schema["priority_enum"]; if "due_date" is present, check it matches
    # schema["due_date_pattern"]. Return (False, <actionable message>) on
    # the first problem found, else (True, None).
    raise NotImplementedError("TODO: implement validate_tool_input")


In [ ]:
print("-" * 60)
print("REPAIR-LOOP DEMO: create_task")
print("-" * 60)

attempt_1 = {"title": "Renew SSL cert", "priority": "urgent", "due_date": "07/30/2026"}
valid, error = validate_tool_input(attempt_1, CREATE_TASK_SCHEMA)
print(f"Attempt 1: {attempt_1}")
print(f"  valid={valid}")
if not valid:
    print(f"  error: {error}")

print()
print("(a model reading that error message has everything it needs to fix")
print(" both problems on its very next turn -- simulating that retry:)")
print()

attempt_2 = {"title": "Renew SSL cert", "priority": "high", "due_date": "2026-07-30"}
valid, error = validate_tool_input(attempt_2, CREATE_TASK_SCHEMA)
print(f"Attempt 2: {attempt_2}")
print(f"  valid={valid}")


In [ ]:
bad_input = {"title": "Renew SSL cert", "priority": "urgent", "due_date": "07/30/2026"}
valid, error = validate_tool_input(bad_input, CREATE_TASK_SCHEMA)
assert valid is False, "bad_input should have failed validation (bad priority AND bad date)"
assert error is not None and "priority" in error.lower(), "error should mention the 'priority' field"
assert "critical" in error or "high" in error, "error should list the valid priority options"

good_input = {"title": "Renew SSL cert", "priority": "high", "due_date": "2026-07-30"}
valid, error = validate_tool_input(good_input, CREATE_TASK_SCHEMA)
assert valid is True and error is None, f"good_input should have passed validation, got error={error!r}"

missing_field = {"priority": "high", "due_date": "2026-07-30"}
valid, error = validate_tool_input(missing_field, CREATE_TASK_SCHEMA)
assert valid is False and "title" in error.lower(), "missing 'title' should be caught and named in the error"

bad_date_only = {"title": "x", "priority": "high", "due_date": "30-07-2026"}
valid, error = validate_tool_input(bad_date_only, CREATE_TASK_SCHEMA)
assert valid is False and "due_date" in error, "wrong date format should be caught and named in the error"

print("Exercise 3 PASSED -- validator catches missing fields, bad enum values,")
print("and bad date formats, each with an actionable, field-specific message.")


## Measuring the Token Bill: Bad vs. Good

Notes Section 8 built this arithmetic for tool *schemas*; the same logic applies to tool *return values* (Section 4). Below, `dump_tasks()` (bad -- no filter, full records) and `search_tasks(status="open", response_format="concise")` (good -- filtered, concise) answer a similar question about the same data. `estimate_tokens` is a simple `len(text) // 4` heuristic (a commonly cited rough approximation, not a real tokenizer) -- good enough to see the shape of the difference, not for a real invoice.

In [ ]:
def estimate_tokens(text):
    """Rough len(text)//4 approximation -- NOT a real tokenizer."""
    return max(1, len(text) // 4)


bad_response = _bad_dump_tasks()
good_response = _good_search_tasks(status="open", response_format="concise")

bad_tokens = estimate_tokens(bad_response)
good_tokens = estimate_tokens(good_response)
reduction_pct = (1 - good_tokens / bad_tokens) * 100

print("-" * 60)
print("TOKEN BILL: dump_tasks() [bad] vs. search_tasks(status='open') [good]")
print("-" * 60)
print(f"  dump_tasks() returned all {len(TASK_STORE)} tasks, full records")
print(f"    ~{bad_tokens} tokens (estimated)")
print()
print(f"  search_tasks(status='open', response_format='concise') returned")
print(f"    the first page of open tasks, one line each")
print(f"    ~{good_tokens} tokens (estimated)")
print()
print(f"  Reduction: {reduction_pct:.0f}%")
print()

N_STEPS = 6
print(f"If a tool this shape were called once per step across a {N_STEPS}-step")
print(f"loop (Chapter 2's growing-tail cost applies to every one of these):")
print(f"  bad:  {N_STEPS} x ~{bad_tokens} tokens  = ~{N_STEPS * bad_tokens} tokens")
print(f"  good: {N_STEPS} x ~{good_tokens} tokens  = ~{N_STEPS * good_tokens} tokens")


## Optional -- Run Both Tool Sets Against a Real Claude Model

Same task prompt, run twice: once with `BAD_TOOL_SCHEMAS`/`BAD_DISPATCH`, once with `GOOD_TOOL_SCHEMAS`/`GOOD_DISPATCH`. This is a genuinely live comparison, not scripted -- whatever the model actually does with each tool set is what gets measured and printed (steps taken, tool-call errors encountered, and cumulative tokens), via `AnthropicBedrockMantle` (Claude Sonnet, through AWS Bedrock).

In [ ]:
RUN_REAL_TOOL_DEMO = False

REAL_DEMO_PROMPT = (
    "Find all open, high-priority tasks assigned to alex, and tell me "
    "their titles and due dates. Use the tools -- do not guess."
)


def _run_one_tool_set(client, model, tools, dispatch, max_steps=6):
    """
    A minimal agent loop (Chapter 2's core cycle, without the budget guard
    or trajectory logging -- this demo only needs step/error/token counts).
    Returns: (final_text_or_None, steps_taken, tool_errors, cumulative_tokens)
    """
    messages = [{"role": "user", "content": REAL_DEMO_PROMPT}]
    cumulative_tokens = 0
    tool_errors = 0

    for step in range(1, max_steps + 1):
        response = client.messages.create(model=model, max_tokens=1024, tools=tools, messages=messages)
        cumulative_tokens += response.usage.input_tokens + response.usage.output_tokens

        if response.stop_reason == "end_turn":
            final_text = next((b.text for b in response.content if b.type == "text"), "")
            return final_text, step, tool_errors, cumulative_tokens

        messages.append({"role": "assistant", "content": response.content})
        results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            fn = dispatch.get(block.name)
            try:
                if fn is None:
                    raise ValueError(f"no such tool '{block.name}'")
                content_str = str(fn(**block.input))
                is_error = False
            except Exception as exc:
                content_str, is_error = f"Error: {exc}", True
            if is_error:
                tool_errors += 1
            results.append({
                "type": "tool_result", "tool_use_id": block.id,
                "content": content_str, "is_error": is_error,
            })
        messages.append({"role": "user", "content": results})

    return None, max_steps, tool_errors, cumulative_tokens


def run_real_tool_demo():
    if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
        print("Skipping real tool demo: AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env.")
        return

    real_client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )

    # A real client raises the SDK's own exception types directly (e.g.
    # AuthenticationError) -- unlike the offline exercises above, nothing
    # here catches or reshapes those, so wrap each run rather than letting
    # a raw traceback surface.
    for label, tools, dispatch in [
        ("BAD tool set", BAD_TOOL_SCHEMAS, BAD_DISPATCH),
        ("GOOD tool set", GOOD_TOOL_SCHEMAS, GOOD_DISPATCH),
    ]:
        print("-" * 60)
        print(f"RUNNING: {label}")
        print("-" * 60)
        try:
            final_text, steps, errors, tokens = _run_one_tool_set(
                real_client, MODEL_NAME, tools, dispatch,
            )
        except Exception as exc:
            print(f"Real run failed: {type(exc).__name__}: {exc}")
            continue
        print(f"  steps={steps}  tool_errors={errors}  cumulative_tokens={tokens}")
        print(f"  final answer: {final_text}")
        print()


if RUN_REAL_TOOL_DEMO:
    run_real_tool_demo()
else:
    print("RUN_REAL_TOOL_DEMO is False -- running in offline/scripted mode only.")
    print("Flip it to True to run both tool sets against a real Claude model via Bedrock.")


## Key Takeaways

You've now seen, concretely, that a tool's name, its argument shape, its return-value size, and its error text are all part of the same interface contract -- and that fixing all four (this chapter's bad-to-good tool set) requires no change to the underlying data or capability at all. You've implemented a truncate-with-cursor return (Exercise 2) and a validation-repair error message (Exercise 3), the same two patterns notes Sections 4 and 5 named directly. And you've measured, not just read about, the token cost of a verbose, unfiltered return value against a lean, paginated one.

**Connection forward:** Chapter 4 takes Section 8's token-bill arithmetic and generalizes it past tool schemas into the entire context window -- system prompt, tool schemas, memory, retrieved documents, and the accumulating transcript -- as one managed, finite resource, and introduces the four operations (write, select, compress, isolate) that every context-management technique from here forward turns out to be one instance of.